This example shows how to compare two Feff Models.  It follows up on the example in  
      Feff_ZnSE_Part1_Running_Feff.ipynb

which must be run before this one.

First, we will read in the data for ZnSe from that example and re-calculate the Fourier transform.

Second, we will set up Feffit models for both cubic and hexagonal structures of ZnSe.  As we saw in the previous example, the local structures were pretty similar.  

Third, we will run and compare the two fits, to see which if we can say which of the two structural models better matches the data.

This example uses Bokeh for plotting, which produces interactive plots in the Jupyter notebook.

All of these steps can also be done in the Larix GUI.  As with the previous example, we'll follow some of the conventions used in Larix so that this can be used as a guide for going back and forth between using Larix and doing similar work programmatically.

This example will use the Zn K-edge of ZnSe data.

In [8]:
# step 0: some basic imports
import os
import numpy as np
from larch.io import read_ascii
from larch.fitting import param_group, param
from larch.xafs import pre_edge, autobk, xftf, feffpath, feffit, feffit_transform, feffit_dataset, feffit_report
# import plotting functions for Bokenh
from larch.plot.bokeh_xafsplots import (plot_chik, plot_chir, plot_chifit, multi_plot, plotlabels, set_label_weight)

from pathlib import Path
from larch import site_config

feff_folder = Path(site_config.user_larchdir, 'feff')

In [11]:
# step 1: read in the ZnSe data as in the previous example

znse_data  = read_ascii(Path('..', 'xafsdata', 'znse_chi.dat'))

# optionally:
# znse_data  = read_ascii(Path('..', 'xafsdata', 'znse_zn_xafs.001'), labels=['energy', 'time', 'i0', 'itrans'])
# znse_data.mu = -np.log(znse_data.itrans/znse_data.i0)
# pre_edge(znse_data)
# autobk(znse_data, rbkg=1.1, kweight=2)

xftf(znse_data, kmin=2, kmax=13, dk=4, kweight=2, window='hanning')

plot_chik(znse_data, kweight=2)
plot_chir(znse_data, show_real=True, rmax=7.5)

['array_labels', 'array_labels_orig', 'attrs', 'chi', 'data', 'delta_chi', 'filename', 'header', 'k', 'path']


Loading BokehJS ...

Loading BokehJS ...

In [12]:
# step 2: build up a Model for cubic ZnSe

trans = feffit_transform(kmin=2.5, kmax=13.5, kw=2, dk=4, window='hanning', rmin=1.2, rmax=4.8)

# For the cubic phase the paths are (hint: Larix shows this well!)
#
# Path filename      Distance   Npaths  Importance   Atoms
#   feff0001.dat        2.45       4     100.0     [Zn]-Se-[Zn]
#   feff0002.dat        4.01      12     100.0     [Zn]-Zn-[Zn]
#   feff0003.dat        4.46      12       9.4     [Zn]-Se-Se-[Zn]
#   feff0004.dat        4.46      24      41.2     [Zn]-Zn-Se-[Zn]
#   feff0005.dat        4.70      12      64.8     [Zn]-Se-[Zn]
#
#  after this, there are some 4 multiple scattering paths, but those have pretty 
#  low "Importance" (below 10), so I am going to ignore them for now.

# we will make a set of Parameters for the cubic model
cpars = param_group(s02 = param(1,   vary=True), 
                    e0  = param(0.1, vary=True),
                    delr_Se245 = param(.002, vary=True),
                    sig2_Se245 = param(.002, vary=True),
                    delr_Zn401 = param(.002, vary=True),
                    sig2_Zn401 = param(.002, vary=True),
                    delr_ZnSe446 = param(.002, vary=True),
                    sig2_ZnSe446 = param(.002, vary=True),
                    delr_Se470 = param(.002, vary=True),
                    sig2_Se470 = param(.002, vary=True),
                      )
cfolder = Path(feff_folder, 'ZnSe_cubic')
cpath_Se245    = feffpath(cfolder/'feff0001.dat', s02='s02', e0='e0', deltar='delr_Se245', sigma2='sig2_Se245')
cpath_Zn401    = feffpath(cfolder/'feff0002.dat', s02='s02', e0='e0', deltar='delr_Zn401', sigma2='sig2_Zn401')
cpath_SeSe446  = feffpath(cfolder/'feff0003.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')
cpath_ZnSe446  = feffpath(cfolder/'feff0004.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')
cpath_Se470    = feffpath(cfolder/'feff0005.dat', s02='s02', e0='e0', deltar='delr_Se470', sigma2='sig2_Se470')
 
# note that we used the same parameters for 'deltar' and 'sigma2' for the two triangle paths at 4.46 Ang.

# define dataset to include data, pathlist, transform
cdataset  = feffit_dataset(data=znse_data, transform=trans,
                           pathlist=[cpath_Se245, cpath_Zn401, cpath_SeSe446, cpath_ZnSe446, cpath_Se470])

# perform fit
cresult = feffit(cpars, cdataset)

print(feffit_report(cresult))

plot_chifit(cdataset, title='Five Paths for cubic ZnSe', show_real=True, rmax=6)

=================== FEFFIT RESULTS ====================
[[Statistics]]
  n_function_calls     = 167
  n_variables          = 10
  n_data_points        = 234
  n_independent        = 26.2101430
  chi_square           = 216.991688
  reduced chi_square   = 13.3861674
  r-factor             = 0.00427580
  Akaike info crit     = 75.4007088
  Bayesian info crit   = 88.0621736
 
[[Parameters]]
  delr_Se245           = -0.0035403 +/- 0.0022415  (init= 0.0020000)
  delr_Se470           =  0.0213558 +/- 0.0112345  (init= 0.0020000)
  delr_Zn401           =  0.0128294 +/- 0.0090743  (init= 0.0020000)
  delr_ZnSe446         =  0.0165435 +/- 0.0644000  (init= 0.0020000)
  e0                   =  5.4558108 +/- 0.4808215  (init= 0.1000000)
  s02                  =  0.9698375 +/- 0.0406066  (init= 1.0000000)
  sig2_Se245           =  0.0063702 +/- 2.7971e-4  (init= 0.0020000)
  sig2_Se470           =  0.0186807 +/- 0.0014986  (init= 0.0020000)
  sig2_Zn401           =  0.0200991 +/- 0.0012771  (init= 

Loading BokehJS ...

Loading BokehJS ...

(<larch.plot.bokeh_xafsplots.BokehFigure at 0x1b240baf0>,
 <larch.plot.bokeh_xafsplots.BokehFigure at 0x1b2a8a970>)

In [13]:
# step 3: now try 

# For the hexagonal phase, there are more paths to include
#
# Path filename      Distance   Npaths  Importance   Atoms
#   feff0001.dat        2.42       3     100.0     [Zn]-Se-[Zn]
#   feff0002.dat        2.51       1      30.7     [Zn]-Se-[Zn]
#   feff0003.dat        3.98       6      65.5     [Zn]-Zn-[Zn]
#   feff0004.dat        3.99       6      65.0     [Zn]-Zn-[Zn]
#   feff0005.dat        4.01       1      10.3     [Zn]-Se-[Zn]
#   feff0006.dat        4.41       6       6.4     [Zn]-Se-Se-[Zn]
#   feff0007.dat        4.41      12      28.9     [Zn]-Zn-Se-[Zn]
#   feff0008.dat        4.46       6       5.5     [Zn]-Se-Se-[Zn]
#   feff0009.dat        4.46       6      12.3     [Zn]-Zn-Se-[Zn]
#   feff0010.dat        4.46       6      12.3     [Zn]-Zn-Se-[Zn]
#   feff0011.dat        4.66       3      21.3     [Zn]-Se-[Zn]
#   feff0012.dat        4.71       6      41.5     [Zn]-Se-[Zn]

#  again, past this there are some 4 multiple scattering paths, with "Importance" below 10.



# we will make a set of Parameters for the cubic model
hpars = param_group(s02 = param(1,   vary=True), 
                    e0  = param(0.1, vary=True),
                    delr_Se242 = param(.002, vary=True),
                    sig2_Se242 = param(.002, vary=True),
                    delr_Zn400 = param(.002, vary=True),
                    sig2_Zn400 = param(.002, vary=True),
                    delr_ZnSe446 = param(.002, vary=True),
                    sig2_ZnSe446 = param(.002, vary=True),
                    delr_Se470 = param(.002, vary=True),
                    sig2_Se470 = param(.002, vary=True),
                      )
hfolder = Path(feff_folder, 'ZnSe_hexag')

# note that we used the same parameters for 'deltar' and 'sigma2' for paths with near-identical distance.

hpath_Se242    = feffpath(hfolder/'feff0001.dat', s02='s02', e0='e0', deltar='delr_Se242', sigma2='sig2_Se242')
hpath_Se251    = feffpath(hfolder/'feff0002.dat', s02='s02', e0='e0', deltar='delr_Se242', sigma2='sig2_Se242')

hpath_Zn398    = feffpath(hfolder/'feff0003.dat', s02='s02', e0='e0', deltar='delr_Zn400', sigma2='sig2_Zn400')
hpath_Zn399    = feffpath(hfolder/'feff0004.dat', s02='s02', e0='e0', deltar='delr_Zn400', sigma2='sig2_Zn400')
hpath_Se401    = feffpath(hfolder/'feff0005.dat', s02='s02', e0='e0', deltar='delr_Zn400', sigma2='sig2_Zn400')


hpath_SeSe441  = feffpath(hfolder/'feff0006.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')
hpath_ZnSe441  = feffpath(hfolder/'feff0007.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')

hpath_SeSe446  = feffpath(hfolder/'feff0008.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')
hpath_ZnSe446a = feffpath(hfolder/'feff0009.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')
hpath_ZnSe446b = feffpath(hfolder/'feff0010.dat', s02='s02', e0='e0', deltar='delr_ZnSe446', sigma2='sig2_ZnSe446')

hpath_Se466    = feffpath(hfolder/'feff0011.dat', s02='s02', e0='e0', deltar='delr_Se470', sigma2='sig2_Se470')
hpath_Se471    = feffpath(hfolder/'feff0012.dat', s02='s02', e0='e0', deltar='delr_Se470', sigma2='sig2_Se470')
 
# note that we used the same parameters for 'deltar' and 'sigma2' for the two triangle paths at 4.46 Ang.

# define dataset to include data, pathlist, transform
hdataset  = feffit_dataset(data=znse_data, transform=trans,
                           pathlist=[hpath_Se242, hpath_Se251, hpath_Zn398, hpath_Zn399, hpath_Se401, hpath_SeSe441, hpath_ZnSe441, 
                                     hpath_SeSe446, hpath_ZnSe446a, hpath_ZnSe446b, hpath_Se466, hpath_Se471 ])

# perform fit
hresult = feffit(hpars, hdataset)

print(feffit_report(hresult))

plot_chifit(hdataset, title='Twelve Paths for hexagonal ZnSe', show_real=True, rmax=6)



=================== FEFFIT RESULTS ====================
[[Statistics]]
  n_function_calls     = 133
  n_variables          = 10
  n_data_points        = 234
  n_independent        = 26.2101430
  chi_square           = 243.385153
  reduced chi_square   = 15.0143742
  r-factor             = 0.00479588
  Akaike info crit     = 78.4092698
  Bayesian info crit   = 91.0707345
 
[[Parameters]]
  delr_Se242           =  0.0192561 +/- 0.0021979  (init= 0.0020000)
  delr_Se470           =  0.0447602 +/- 0.0140204  (init= 0.0020000)
  delr_Zn400           =  0.0284969 +/- 0.0133141  (init= 0.0020000)
  delr_ZnSe446         =  0.0170287 +/- 0.0393602  (init= 0.0020000)
  e0                   =  6.6619101 +/- 0.4602741  (init= 0.1000000)
  s02                  =  0.9752756 +/- 0.0429562  (init= 1.0000000)
  sig2_Se242           =  0.0047492 +/- 2.9442e-4  (init= 0.0020000)
  sig2_Se470           =  0.0165152 +/- 0.0017471  (init= 0.0020000)
  sig2_Zn400           =  0.0191156 +/- 0.0018572  (init= 

Loading BokehJS ...

Loading BokehJS ...

(<larch.plot.bokeh_xafsplots.BokehFigure at 0x1b19aaa90>,
 <larch.plot.bokeh_xafsplots.BokehFigure at 0x1aa3981d0>)

In [7]:
# The results are pretty close... from the fit statistics:
#
#  Statistic              Cubic          Hexagonal
#  chi_square           = 216.991688     243.385153
#  reduced chi_square   = 13.3861674     15.0143742 
#  r-factor             = 0.00427580     0.00479588
#  Akaike info crit     = 75.4007088     78.4092698
#  Bayesian info crit   = 88.0621736     91.0707345

# The two models use the same number of parameters.
# Since the cubic model uses fewer paths and gives marginally better values for 
# all fit statistics, it would probably be preferred, but maybe not "proven". 